In [13]:
import cv2
import numpy as np

cap = cv2.VideoCapture("scube.mp4")
feature_params = dict(maxCorners=400,
                      qualityLevel=0.1,
                      minDistance=3,
                      blockSize=7)
lk_params = dict(winSize=(31, 31),
                 maxLevel=3,
                 criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 15, 0.03))
ret, old_frame = cap.read()
old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)
mask = np.zeros_like(old_frame)
good_new = None 
for i in range(200):
    ret, frame = cap.read()
    if not ret:
        break
    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    if p0 is not None:
        p1, st, err = cv2.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None, **lk_params)
        if p1 is not None:
            good_new = p1[st == 1]
            good_old = p0[st == 1]
            for new, old in zip(good_new, good_old):
                a, b = new.ravel()
                c, d = old.ravel()
                mask = cv2.line(mask, (int(a), int(b)), (int(c), int(d)), (0,255,0), 2)
                frame = cv2.circle(frame, (int(a), int(b)), 5, (0,0,255), -1)
    if (good_new is None) or (len(good_new) < 50) or (i == 28):
        p0 = cv2.goodFeaturesToTrack(frame_gray, mask=None, **feature_params)
    else:
        p0 = good_new.reshape(-1, 1, 2)
    img = cv2.add(frame, mask)
    img = cv2.resize(img, (600, 800))
    cv2.imshow("LK flow", img)
    old_gray = frame_gray.copy()
    if cv2.waitKey(30) & 0xFF == 27:
        break
cap.release()
cv2.destroyAllWindows()

In [15]:
cap = cv2.VideoCapture("scube.mp4")
ret, frame1 = cap.read()
prev = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)

for i in range(200):
    ret, frame2 = cap.read()
    if not ret:
        break
    next = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)
    # prev = cv2.GaussianBlur(prev, (5,5), 0)
    # next = cv2.GaussianBlur(next, (5,5), 0)
    flow = cv2.calcOpticalFlowFarneback(
        prev, next, None,
        pyr_scale=0.5,
        levels=4,
        winsize=25,
        iterations=5,
        poly_n=7,
        poly_sigma=1.5,
        flags=cv2.OPTFLOW_FARNEBACK_GAUSSIAN
    )
    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    hsv = np.zeros_like(frame2)
    hsv[..., 1] = 255
    hsv[..., 0] = ang * 180 / np.pi / 2
    hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
    rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    rgb = cv2.resize(rgb, (600, 800))
    _, motion_mask = cv2.threshold(mag, 2.0, 255, cv2.THRESH_BINARY)
    motion_mask = cv2.resize(motion_mask, (600, 800))
    # kernel = np.ones((5,5), np.uint8)
    # motion_mask = cv2.morphologyEx(motion_mask.astype(np.uint8), cv2.MORPH_OPEN, kernel)
    # motion_mask = cv2.morphologyEx(motion_mask, cv2.MORPH_DILATE, kernel)
    cv2.imshow("dense", rgb)
    cv2.imshow("mask", motion_mask.astype(np.uint8))
    prev = next

    if cv2.waitKey(30) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()